In [25]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *

## Geometry and mesh generation

In [26]:
rect = WorkPlane().RectangleC(10,10).Face()
rect.name = "wall"
mesh = rect.GenerateMesh(maxh=0.05, dim=2)
Draw (mesh);

In [27]:
fes = H1(mesh, order=4, complex=True, dirichlet="wall")
u, v = fes.TnT()  

In [28]:
dt = 0.01
t_end = 4.0
steps = int(t_end / dt)


In [29]:
V = CF(0)


In [30]:
m_form = BilinearForm(fes)
m_form += u * v * dx
m_form.Assemble()


In [31]:
h_form = BilinearForm(fes)
h_form += (0.5 * grad(u) * grad(v) + V * u * v) * dx
h_form.Assemble()


In [32]:
# Center (x0, y0) = (-2, 0), Width sigma = 0.7, Momentum kx = 3.0 (moves right)
x0, y0 = -2.0, 0.0
sigma = 0.5
kx, ky = 10.0, 0.0

psi_init = exp(-((x-x0)**2 + (y-y0)**2) / (2 * sigma**2)) * exp(1j * (kx*x + ky*y))


In [33]:
gfu = GridFunction(fes)
gfu.Set(psi_init)


In [34]:
# Left-Hand Side (LHS) matrix: M + i * (dt/2) * H
lhs_form = BilinearForm(fes)
lhs_form += (u * v + 1j * (dt / 2) * (0.5 * grad(u) * grad(v) + V * u * v)) * dx
lhs_form.Assemble()

lhs_inv = lhs_form.mat.Inverse(freedofs=fes.FreeDofs(), inverse="sparsecholesky")
rhs_mat = m_form.mat - 1j * (dt / 2) * h_form.mat


In [35]:
scenes = []
scenes.append(Draw(gfu, mesh, "psi"))
# We want to look at the probability density |psi|^2
prob_density = Conj(gfu) * gfu
scenes.append(Draw(prob_density, mesh, "Probability_Density"))


In [36]:
rhs_vec = gfu.vec.CreateVector()

t = 0.0

print("Starting simulation... (Close the GUI window or press Ctrl+C to stop)")

with TaskManager():  # Optimizes parallel execution in NGSolve
    for step in range(steps):
        # Action of RHS matrix on the current state: rhs_vec = (M - i*dt/2*H) * u
        rhs_vec.data = rhs_mat * gfu.vec
        
        # Solve the linear system: u_next = LHS^-1 * rhs_vec
        gfu.vec.data = lhs_inv * rhs_vec
        
        t += dt
        
        # Redraw the solution in the GUI scene
        if step % 10 == 0:  # Update the GUI every 10 steps for performance
            for scene in scenes:
                scene.Redraw()
        


Starting simulation... (Close the GUI window or press Ctrl+C to stop)
